# 03 Calibration And CI

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import zipfile

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

try:
    from google.colab import drive
    if not _drive_root_ready(DRIVE_ROOT):
        try:
            drive.mount(str(DRIVE_MOUNT))
        except Exception as exc:
            print(f'Drive mount initial attempt failed or was stale: {exc}')
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
    else:
        print('Drive root already visible:', DRIVE_ROOT)
except Exception as exc:
    print(f'Drive mount skipped or unavailable: {exc}')

if not _drive_root_ready(DRIVE_ROOT):
    raise RuntimeError(f'Google Drive root is not readable at {DRIVE_ROOT}. Restart/remount before continuing.')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA'))
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
# Notebook 03 consumes frozen artifacts only. Dataset/model discovery is
# advisory and must never force a GPU runtime or a multi-GB archive restore.
chapman_candidates = [
    DRIVE_ROOT / 'WFDB-ChapmanShaoxing.zip',
    DRIVE_ROOT / 'WFDB_ChapmanShaoxing.zip',
    DRIVE_ROOT / 'chapman.zip',
    DRIVE_ROOT / 'archive.zip',
]
chapman_zip = next((path for path in chapman_candidates if path.is_file()), None)
if chapman_zip is not None:
    if chapman_zip.stat().st_size == 0 or not zipfile.is_zipfile(chapman_zip):
        raise ValueError(f'Visible Chapman archive is invalid: {chapman_zip}')
    os.environ['ECG_RAMBA_CHAPMAN_ZIP'] = str(chapman_zip)
else:
    print('Chapman archive is not visible; this is valid for artifact-only calibration.')

model_pointer = DRIVE_ROOT / 'model_runs' / 'current_final_ema_model_dir.txt'
model_dir_env = os.environ.get('ECG_RAMBA_MODEL_DIR', '').strip()
model_dir = Path(model_dir_env).expanduser() if model_dir_env else None
if model_dir is None and model_pointer.is_file():
    pointer_value = model_pointer.read_text(encoding='utf-8').strip()
    model_dir = Path(pointer_value).expanduser() if pointer_value else None
if model_dir is not None and model_dir.is_dir():
    os.environ['ECG_RAMBA_MODEL_DIR'] = str(model_dir)
else:
    model_dir = None
    print('Model directory is not visible; this is valid for artifact-only calibration.')

os.environ['ECG_RAMBA_LOCAL_ROOT'] = str(LOCAL_RUNTIME_ROOT)
os.environ['ECG_RAMBA_EXTRACT_DIR'] = str(LOCAL_RUNTIME_ROOT / 'chapman')
os.environ['ECG_RAMBA_USE_CLEAN_CACHE'] = '0'
os.environ['ECG_RAMBA_SAVE_CLEAN_CACHE'] = '0'

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
        if os.environ.get('ECG_RAMBA_ALLOW_STALE_REPO', '0') != '1':
            raise RuntimeError('git pull failed; refusing to continue with stale code. Set ECG_RAMBA_ALLOW_STALE_REPO=1 only for debugging.')
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)

# BEGIN FORENSIC CODE AUTHORITY PIN
FORENSIC_CODE_AUTHORITY_CAPABILITY = 'canonical_git_commit_pin_v1'
FORENSIC_CODE_AUTHORITY_SCHEMA_VERSION = 1
_AUTHORITY_BOOTSTRAP_ALLOWED = False

def _pin_forensic_code_authority():
    import json as _authority_json
    import os as _authority_os
    import re as _authority_re
    import subprocess as _authority_subprocess
    import uuid as _authority_uuid
    from datetime import datetime as _authority_datetime, timezone as _authority_timezone
    from pathlib import Path as _AuthorityPath
    from scripts.revision.artifact_mirror import PublishLock as _AuthorityPublishLock

    manifest_path = _AuthorityPath(MIRROR_REVISION_ROOT) / 'manifests' / 'notebook_code_authority.json'
    requested_commit = _authority_os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT', '').strip().lower()
    reset_requested = _authority_os.environ.get('ECG_RAMBA_RESET_CODE_AUTHORITY', '0') == '1'
    legacy_rotate_requested = (
        _authority_os.environ.get('ECG_RAMBA_ROTATE_CODE_AUTHORITY_TO_BRANCH_HEAD', '0') == '1'
    )
    if legacy_rotate_requested:
        raise RuntimeError(
            'Implicit authority rotation to a moving branch head is disabled. '
            'Set ECG_RAMBA_RESET_CODE_AUTHORITY=1 together with an explicit full '
            'ECG_RAMBA_AUTHORITY_COMMIT in Notebook 00.'
        )
    rotate_to_branch_head = False
    commit_pattern = _authority_re.compile(r'[0-9a-f]{40}')

    def git(*args, check=True):
        result = _authority_subprocess.run(
            ['git', *args],
            cwd=str(REPO_DIR),
            check=False,
            text=True,
            stdout=_authority_subprocess.PIPE,
            stderr=_authority_subprocess.STDOUT,
        )
        if check and result.returncode:
            raise RuntimeError(
                'Code-authority git command failed: git '
                + ' '.join(args)
                + '\n'
                + (result.stdout or '')[-4000:]
            )
        return result

    if reset_requested and not _AUTHORITY_BOOTSTRAP_ALLOWED:
        raise RuntimeError(
            'Only Notebook 00 may rotate the canonical code authority. '
            'Run Notebook 00 with ECG_RAMBA_RESET_CODE_AUTHORITY=1 and an explicit '
            'ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    if reset_requested and not commit_pattern.fullmatch(requested_commit):
        raise RuntimeError(
            'Authority reset requires ECG_RAMBA_AUTHORITY_COMMIT as a full 40-character git SHA.'
        )

    manifest = None
    with _AuthorityPublishLock(
        _AuthorityPath(MIRROR_REVISION_ROOT),
        run_id='authority-read-' + _authority_uuid.uuid4().hex,
    ):
        if manifest_path.is_file() and not reset_requested:
            manifest = _authority_json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest is not None:
        if manifest.get('capability') != 'canonical_git_commit_pin_v1':
            raise RuntimeError('Canonical code-authority manifest capability is invalid.')
        if int(manifest.get('schema_version', 0)) != 1:
            raise RuntimeError('Canonical code-authority manifest schema is invalid.')
        expected_commit = str(manifest.get('git_commit', '')).strip().lower()
        if not commit_pattern.fullmatch(expected_commit):
            raise RuntimeError('Canonical code-authority manifest lacks a full git SHA.')
        if str(manifest.get('repository_url', '')).rstrip('/') != str(REPO_URL).rstrip('/'):
            raise RuntimeError('Canonical code-authority repository URL differs from this notebook.')
        if str(manifest.get('branch', '')) != str(BRANCH):
            raise RuntimeError('Canonical code-authority branch differs from this notebook runtime.')
        if requested_commit and requested_commit != expected_commit:
            raise RuntimeError(
                'ECG_RAMBA_AUTHORITY_COMMIT differs from the canonical authority manifest. '
                'Rotate authority explicitly in Notebook 00; do not override it in a downstream notebook.'
            )
    else:
        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            raise FileNotFoundError(
                'Canonical code-authority manifest is missing. Run Notebook 00 first in a fresh runtime; '
                'downstream notebooks fail closed instead of following a moving branch.'
            )
        expected_commit = requested_commit or git('rev-parse', 'HEAD').stdout.strip().lower()
        if not commit_pattern.fullmatch(expected_commit):
            raise RuntimeError('Notebook 00 could not resolve a full code-authority git SHA.')

    tracked_status = git('status', '--porcelain', '--untracked-files=no').stdout.strip()
    if tracked_status:
        raise RuntimeError(
            'Tracked files differ from git before authority checkout. Use a fresh Colab clone; '
            'authority pinning will not stash or overwrite local edits.\n' + tracked_status[:4000]
        )

    fetch = git('fetch', 'origin', '--prune', check=False)
    if fetch.returncode:
        print('WARNING: git fetch failed; accepting only an already-present pinned commit.')
        print((fetch.stdout or '')[-2000:])
    git('cat-file', '-e', expected_commit + '^{commit}')
    git('checkout', '--detach', expected_commit)
    observed_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
    if observed_commit != expected_commit:
        raise RuntimeError(
            f'Code-authority checkout mismatch: expected={expected_commit} observed={observed_commit}'
        )

    if manifest is None or reset_requested:
        manifest = {
            'capability': 'canonical_git_commit_pin_v1',
            'schema_version': 1,
            'git_commit': expected_commit,
            'repository_url': str(REPO_URL),
            'branch': str(BRANCH),
            'established_utc': _authority_datetime.now(_authority_timezone.utc).isoformat(),
            'established_by': '00_colab_bootstrap.ipynb',
            'selection': (
                'explicit_environment_sha'
                if requested_commit
                else 'fetched_branch_head_at_initial_bootstrap'
            ),
        }
        with _AuthorityPublishLock(
            _AuthorityPath(MIRROR_REVISION_ROOT),
            run_id='authority-write-' + _authority_uuid.uuid4().hex,
        ):
            if manifest_path.is_file() and not reset_requested:
                concurrent = _authority_json.loads(manifest_path.read_text(encoding='utf-8'))
                if concurrent != manifest:
                    raise RuntimeError('A concurrent runtime established a different code authority.')
            else:
                manifest_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = manifest_path.with_name(
                    manifest_path.name + '.partial.' + _authority_uuid.uuid4().hex
                )
                with temporary.open('w', encoding='utf-8') as handle:
                    handle.write(_authority_json.dumps(manifest, indent=2, sort_keys=True) + '\n')
                    handle.flush()
                    _authority_os.fsync(handle.fileno())
                _authority_os.replace(temporary, manifest_path)
        print('Established canonical code authority:', manifest_path)

    _authority_os.environ['ECG_RAMBA_AUTHORITY_COMMIT'] = expected_commit
    _authority_os.environ.pop('ECG_RAMBA_RESET_CODE_AUTHORITY', None)
    globals()['CODE_AUTHORITY_MANIFEST_PATH'] = manifest_path
    globals()['CODE_AUTHORITY'] = manifest
    print('Pinned code authority:', expected_commit)
    print('Authority manifest   :', manifest_path)
    return manifest

CODE_AUTHORITY = _pin_forensic_code_authority()
# END FORENSIC CODE AUTHORITY PIN

_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('chapman_zip:', chapman_zip, '| size=', chapman_zip.stat().st_size)
print('model_dir  :', model_dir)
print('model pointer:', model_pointer, 'exists=', model_pointer.is_file())
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        log_relative = log_path.resolve().relative_to((REPO_DIR / 'reports' / 'revision').resolve())
    except ValueError:
        log_relative = Path('logs') / log_path.name
    durable_log_path = MIRROR_REVISION_ROOT / log_relative
    durable_log_path.parent.mkdir(parents=True, exist_ok=True)

    from contextlib import ExitStack
    with ExitStack() as stack:
        log_file = stack.enter_context(log_path.open('a', encoding='utf-8'))
        durable_log_file = stack.enter_context(durable_log_path.open('a', encoding='utf-8'))
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
            durable_log_file.write(line)
            durable_log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    print(f'Durable command log: {durable_log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    return_code = -1
    caught = None
    completed = None
    process = None
    try:
        process = _forensic_subprocess.Popen(
            cmd,
            shell=isinstance(cmd, str),
            cwd=str(run_cwd),
            stdout=_forensic_subprocess.PIPE,
            stderr=_forensic_subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        with history_log.open('a', encoding='utf-8') as local_handle:
            canonical_handle = (
                canonical_history.open('a', encoding='utf-8')
                if canonical_history is not None
                else None
            )
            try:
                for line in process.stdout or ():
                    print(line, end='', flush=True)
                    local_handle.write(line)
                    local_handle.flush()
                    if canonical_handle is not None:
                        canonical_handle.write(line)
                        canonical_handle.flush()
                return_code = int(process.wait())
                local_handle.flush()
                _forensic_os.fsync(local_handle.fileno())
                if canonical_handle is not None:
                    canonical_handle.flush()
                    _forensic_os.fsync(canonical_handle.fileno())
            finally:
                if canonical_handle is not None:
                    canonical_handle.close()
        completed = _forensic_subprocess.CompletedProcess(cmd, return_code)
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except Exception:
                process.kill()
                process.wait()
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed


## Install Metric Dependencies

In [ ]:
INSTALL_METRIC_DEPS = True
if INSTALL_METRIC_DEPS:
    import importlib.util
    import subprocess
    import sys

    required = {
        'numpy': 'numpy',
        'pandas': 'pandas',
        'scipy': 'scipy',
        'sklearn': 'scikit-learn',
        'threadpoolctl': 'threadpoolctl',
        'matplotlib': 'matplotlib',
        'dateutil': 'python-dateutil',
    }
    missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
    print('Python:', sys.version)
    if missing:
        print('Installing missing metric dependencies:', missing)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    else:
        print('Metric dependencies already available; no pip install needed.')

    import matplotlib
    import numpy as np
    import scipy
    import sklearn
    print('numpy     :', np.__version__)
    print('scipy     :', scipy.__version__)
    print('sklearn   :', sklearn.__version__)
    print('matplotlib:', matplotlib.__version__)
else:
    print('Skipping metric dependency install.')


## Restore And Validate Frozen OOF Inputs

In [ ]:
stable_mirror = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
CALIBRATION_RESTORE_RELATIVE_PATHS = [
    'predictions/oof_final_ema_predictions.npz',
    'predictions/oof_final_ema_slice_predictions.npz',
    'metrics/oof_final_ema_prediction_summary.json',
    'tables/oof_final_ema_class_summary.csv',
    'manifests/oof_final_ema_prediction_run_manifest.json',
    'manifests/oof_final_ema_freeze_manifest.json',
    'manifests/oof_final_ema_group_sidecar.npz',
    'metrics/calibration_ci_oof_final_ema_predictions.json',
    'metrics/pooling_sensitivity.csv',
    'manifests/fold_pca_manifest.json',
    'metrics/paired_full_vs_minirocket_comparison.json',
    'metrics/paired_full_vs_resnet_comparison.json',
    'metrics/paired_full_vs_raw_mamba_comparison.json',
    'metrics/paired_full_vs_transformer_comparison.json',
    'metrics/paired_full_vs_hybrid_morphology_comparison.json',
    'figures/figure_calibration_audit.png',
    'tables/table_calibration_ci_compact.csv',
    'tables/table_paired_baseline_ci_compact.csv',
    'tables/table_pooling_sensitivity_compact.csv',
    'tables/table_fold_pca_provenance.csv',
    'tables/table_training_configuration.csv',
    'tables/table_morphology_transform_contract.csv',
    'manifests/reviewer_completion_input_contract.json',
]

def restore_selected_from_mirror(relative_paths):
    import hashlib
    import shutil

    if not stable_mirror.exists():
        print('Stable mirror does not exist yet:', stable_mirror)
        return
    manifest_path = stable_mirror / 'manifests' / 'mirror_manifest.json'
    rows = {}
    if manifest_path.exists():
        mirror_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        rows = {row.get('relative_path'): row for row in mirror_manifest.get('artifacts', []) if row.get('relative_path')}

    def digest(path):
        h = hashlib.sha256()
        with Path(path).open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    restored, reused, unavailable = [], [], []
    destination_root = Path('reports/revision')
    for relative in relative_paths:
        source = stable_mirror / relative
        destination = destination_root / relative
        if not source.exists() or source.stat().st_size == 0:
            unavailable.append(relative)
            continue
        source_sha = digest(source)
        row = rows.get(relative)
        if row is None:
            if destination.exists() and destination.stat().st_size > 0:
                raise RuntimeError(f'Active calibration input is absent from the canonical mirror manifest: {relative}')
            unavailable.append(relative)
            continue
        if row.get('sha256') != source_sha:
            raise RuntimeError(f'Mirror checksum mismatch for {relative}')
        if destination.exists() and destination.stat().st_size > 0 and digest(destination) == source_sha:
            reused.append(relative)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        if digest(destination) != source_sha:
            raise RuntimeError(f'Checksum mismatch after restoring {relative}')
        restored.append(relative)
    print(f'Calibration targeted mirror restore: restored={len(restored)} reused={len(reused)} unavailable={len(unavailable)}')
    if restored:
        print('Restored:', ', '.join(restored))

restore_selected_from_mirror(CALIBRATION_RESTORE_RELATIVE_PATHS)

OOF_STEM = 'oof_final_ema'
OOF_FREEZE = Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json')
freeze_check_cmd = (
    'python -u scripts/revision/06_freeze_oof.py '
    f'--record-file reports/revision/predictions/{OOF_STEM}_predictions.npz '
    f'--slice-file reports/revision/predictions/{OOF_STEM}_slice_predictions.npz '
    f'--summary-file reports/revision/metrics/{OOF_STEM}_prediction_summary.json '
    f'--class-table reports/revision/tables/{OOF_STEM}_class_summary.csv '
    f'--run-manifest reports/revision/manifests/{OOF_STEM}_prediction_run_manifest.json '
    f'--freeze-manifest {OOF_FREEZE} '
    '--expected-records 44186 --expected-folds 5 --q 3 --expected-checkpoint-kind final_ema --manuscript-ready-strict --group-sidecar reports/revision/manifests/oof_final_ema_group_sidecar.npz --check-existing-freeze'
)
run(
    freeze_check_cmd,
    log_path='reports/revision/logs/calibration_oof_final_ema_input_check.log',
)


## Run Calibration And Bootstrap CI

In [ ]:
N_BOOT = 1000
N_BINS = 15
THRESHOLD = 0.5
FORCE_RERUN_CALIBRATION_CI = False
OOF_STEM = 'oof_final_ema'

pred = Path(f'reports/revision/predictions/{OOF_STEM}_predictions.npz')
freeze = Path(f'reports/revision/manifests/{OOF_STEM}_freeze_manifest.json')
out = Path(f'reports/revision/metrics/calibration_ci_{OOF_STEM}_predictions.json')

if not pred.exists() or not freeze.exists():
    raise FileNotFoundError('Notebook 02 must produce and freeze canonical fixed-epoch final_ema OOF artifacts first.')

def _sha256(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

freeze_payload = json.loads(freeze.read_text(encoding='utf-8'))
freeze_group_sidecar = (freeze_payload.get('group_contract') or {}).get('sidecar') or {}
freeze_group_sidecar_path = Path(str(freeze_group_sidecar.get('path', '')))
if freeze_group_sidecar_path and not freeze_group_sidecar_path.is_absolute():
    freeze_group_sidecar_path = Path.cwd() / freeze_group_sidecar_path
if not freeze_group_sidecar_path.is_file():
    raise FileNotFoundError('Authenticated group sidecar is missing: ' + str(freeze_group_sidecar_path))
freeze_group_sidecar_sha = _sha256(freeze_group_sidecar_path)
if freeze_group_sidecar_sha != freeze_group_sidecar.get('sha256'):
    raise RuntimeError('Authenticated group sidecar SHA differs from the strict freeze manifest.')

pred_sha = _sha256(pred)
freeze_sha = _sha256(freeze)
expected_shape = {
    'y_prob': [freeze_payload.get('validated_records'), freeze_payload.get('n_classes')],
    'y_true': [freeze_payload.get('validated_records'), freeze_payload.get('n_classes')],
}

cmd = (
    f'python -u scripts/revision/04_calibration_ci.py '
    f'--predictions "{pred}" --out "{out}" '
    f'--freeze-manifest "{freeze}" --require-manuscript-ready '
    f'--threshold {THRESHOLD} --n-bins {N_BINS} --n-boot {N_BOOT}'
)

stale_reasons = []
if FORCE_RERUN_CALIBRATION_CI:
    stale_reasons.append('FORCE_RERUN_CALIBRATION_CI=True')
elif not out.exists() or out.stat().st_size == 0:
    stale_reasons.append('missing_or_empty_calibration_json')
else:
    try:
        existing = json.loads(out.read_text(encoding='utf-8'))
    except Exception as exc:
        existing = {}
        stale_reasons.append(f'calibration_json_unreadable={exc!r}')
    if existing:
        if existing.get('predictions_sha256') != pred_sha:
            stale_reasons.append(
                'prediction_sha_mismatch: '
                f"json={existing.get('predictions_sha256')} current={pred_sha}"
            )
        if existing.get('freeze_manifest_sha256') != freeze_sha:
            stale_reasons.append(
                'freeze_manifest_sha_mismatch: '
                f"json={existing.get('freeze_manifest_sha256')} current={freeze_sha}"
            )
        if existing.get('shape') != expected_shape:
            stale_reasons.append(f"shape_mismatch: json={existing.get('shape')} expected={expected_shape}")
        if existing.get('threshold') != THRESHOLD:
            stale_reasons.append(f"threshold_mismatch: json={existing.get('threshold')} expected={THRESHOLD}")
        if existing.get('n_bins') != N_BINS:
            stale_reasons.append(f"n_bins_mismatch: json={existing.get('n_bins')} expected={N_BINS}")
        if existing.get('n_boot') != N_BOOT:
            stale_reasons.append(f"n_boot_mismatch: json={existing.get('n_boot')} expected={N_BOOT}")
        bootstrap_contract = existing.get('bootstrap') or {}
        FORENSIC_BOOTSTRAP_BINDING_CHECK = 'freeze_group_sidecar_v1'
        if bootstrap_contract.get('group_semantics_reference') != 'https://physionet.org/content/ecg-arrhythmia/1.0.0/':
            stale_reasons.append('bootstrap_group_semantics_reference_missing_or_mismatched')
        if not bootstrap_contract.get('group_sidecar_sha256'):
            stale_reasons.append('bootstrap_group_sidecar_sha256_missing')
        elif bootstrap_contract.get('group_sidecar_sha256') != freeze_group_sidecar_sha:
            stale_reasons.append('bootstrap_group_sidecar_sha_mismatch')
        if bootstrap_contract.get('unit') != 'authenticated_source_patient_record':
            stale_reasons.append(
                'bootstrap_unit_missing_or_mismatched: '
                f"json={bootstrap_contract.get('unit')} expected=authenticated_source_patient_record"
            )
        if bootstrap_contract.get('independence_contract') != 'physionet_ecg_arrhythmia_one_patient_per_record_v1':
            stale_reasons.append(
                'bootstrap_independence_contract_missing_or_mismatched: '
                f"json={bootstrap_contract.get('independence_contract')} "
                'expected=physionet_ecg_arrhythmia_one_patient_per_record_v1'
            )

if stale_reasons:
    print('Recomputing calibration CI artifact because existing output is missing/stale:')
    for reason in stale_reasons:
        print(' -', reason)
    run(cmd, log_path='reports/revision/logs/oof_final_ema_calibration_ci.log')
else:
    print('Reusing calibration CI artifact; contract matches current frozen OOF and freeze manifest:', out)
    print('  predictions_sha256     :', pred_sha)
    print('  freeze_manifest_sha256 :', freeze_sha)



## Matched Cross-Fitted Calibration Audit


In [ ]:
# CPU-only secondary OOF-score sensitivity audit. Each evaluated fold excludes its labels from calibrator fitting.
# Base models are not nested-refitted; use the generated claim boundary and do not present this as a deploy-time estimate.
MATCHED_CALIBRATION_N_BOOT = 1000
RUN_MATCHED_CALIBRATION_AUDIT = True

matched_runner = Path('scripts/revision/42_matched_oof_calibration.py')
matched_runner_source = matched_runner.read_text(encoding='utf-8', errors='replace') if matched_runner.is_file() else ''
matched_runner_tokens = [
    'PROTOCOL = MATCHED_CALIBRATION_PROTOCOL',
    'cannot reverse within-fold score ordering',
    'fully nested deploy-time calibration estimate',
    '--reuse-bootstrap',
]
missing_matched_runner_tokens = [token for token in matched_runner_tokens if token not in matched_runner_source]
if missing_matched_runner_tokens:
    raise RuntimeError(
        'Matched calibration runner is stale or missing: '
        + ', '.join(missing_matched_runner_tokens)
    )

matched_required_prediction_paths = {
    'full': 'predictions/oof_final_ema_predictions.npz',
    'minirocket': 'predictions/minirocket_only_oof_predictions.npz',
    'resnet': 'predictions/resnet1d_cnn_oof_predictions.npz',
    'raw_mamba': 'predictions/raw_mamba_oof_predictions.npz',
    'transformer': 'predictions/transformer_ecg_oof_predictions.npz',
}
matched_optional_prediction_paths = {
    'frozen_transform_mlp': 'predictions/hybrid_morphology_oof_predictions.npz',
}
restore_selected_from_mirror(
    list(matched_required_prediction_paths.values())
    + list(matched_optional_prediction_paths.values())
)

def require_canonical_matched_input(relative):
    import hashlib

    manifest_path = stable_mirror / 'manifests' / 'mirror_manifest.json'
    if not manifest_path.is_file():
        raise FileNotFoundError(f'Canonical mirror manifest is missing: {manifest_path}')
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    rows = {row.get('relative_path'): row for row in payload.get('artifacts', []) if row.get('relative_path')}
    row = rows.get(relative)
    source = stable_mirror / relative
    active = Path('reports/revision') / relative
    if row is None or not source.is_file() or source.stat().st_size == 0:
        raise FileNotFoundError(f'Matched calibration input is not authenticated by canonical Drive: {relative}')
    def digest(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                value.update(chunk)
        return value.hexdigest()
    expected_sha = str(row.get('sha256') or '')
    expected_size = int(row.get('size_bytes', -1))
    if expected_size != source.stat().st_size or digest(source) != expected_sha:
        raise RuntimeError(f'Canonical mirror contract is invalid for matched input: {relative}')
    if not active.is_file() or active.stat().st_size != expected_size or digest(active) != expected_sha:
        raise RuntimeError(f'Active matched input differs from canonical Drive: {relative}')
    return {'relative_path': relative, 'sha256': expected_sha, 'size_bytes': expected_size}

matched_input_attestations = {}
missing_required_matched_inputs = []
for name, relative in matched_required_prediction_paths.items():
    try:
        matched_input_attestations[name] = require_canonical_matched_input(relative)
    except FileNotFoundError as error:
        missing_required_matched_inputs.append(str(error))
for name, relative in matched_optional_prediction_paths.items():
    try:
        matched_input_attestations[name] = require_canonical_matched_input(relative)
    except FileNotFoundError:
        print(f'Optional matched calibration comparator deferred: {name} ({relative})')
if missing_required_matched_inputs:
    RUN_MATCHED_CALIBRATION_AUDIT = False
    print(
        'Deferred matched calibration audit until Notebook 04 publishes the required OOF baselines:\n - '
        + '\n - '.join(missing_required_matched_inputs)
    )
else:
    print('Canonical matched calibration inputs authenticated:', matched_input_attestations)

matched_model_args = ' '.join(
    f"--model {name}=reports/revision/{contract['relative_path']}"
    for name, contract in matched_input_attestations.items()
)
matched_calibration_command = (
    'python -u scripts/revision/42_matched_oof_calibration.py '
    f'{matched_model_args} --n-boot {MATCHED_CALIBRATION_N_BOOT} '
    '--strict --reuse-bootstrap'
)
print('Matched calibration command:', matched_calibration_command)
if RUN_MATCHED_CALIBRATION_AUDIT:
    run(
        matched_calibration_command,
        log_path='reports/revision/logs/matched_oof_calibration.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
        f'--refresh-existing-prefix metrics/matched_calibration_metric_cache '
        f'--refresh-existing-prefix metrics/matched_oof_calibration_ '
        f'--refresh-existing-prefix tables/table_matched_oof_calibration '
        f'--refresh-existing-prefix tables/table_paired_matched_oof_calibration.csv '
        f'--refresh-existing-prefix figures/figure_matched_calibration_audit.png '
        f'--refresh-existing-prefix manifests/matched_oof_calibration_manifest.json '
        f'--refresh-existing-prefix predictions/full_cross_fitted_platt_oof_predictions.npz '
        f'--refresh-existing-prefix predictions/minirocket_cross_fitted_platt_oof_predictions.npz '
        f'--refresh-existing-prefix predictions/resnet_cross_fitted_platt_oof_predictions.npz '
        f'--refresh-existing-prefix predictions/raw_mamba_cross_fitted_platt_oof_predictions.npz '
        f'--refresh-existing-prefix predictions/transformer_cross_fitted_platt_oof_predictions.npz '
        f'--refresh-existing-prefix predictions/frozen_transform_mlp_cross_fitted_platt_oof_predictions.npz '
        f'--mirror-root "{MIRROR_REVISION_ROOT}"',
        log_path='reports/revision/logs/matched_oof_calibration_mirror_publish.log',
    )

matched_outputs = [
    Path('reports/revision/metrics/matched_oof_calibration_summary.json'),
    Path('reports/revision/metrics/matched_oof_calibration_bootstrap.json'),
    Path('reports/revision/tables/table_matched_oof_calibration.csv'),
    Path('reports/revision/tables/table_matched_oof_calibration_coefficients.csv'),
    Path('reports/revision/tables/table_matched_oof_calibration.tex'),
    Path('reports/revision/tables/table_paired_matched_oof_calibration.csv'),
    Path('reports/revision/figures/figure_matched_calibration_audit.png'),
    Path('reports/revision/manifests/matched_oof_calibration_manifest.json'),
]
for path in matched_outputs:
    print(path, 'exists=', path.is_file(), 'size=', path.stat().st_size if path.is_file() else None)
if not RUN_MATCHED_CALIBRATION_AUDIT:
    print(
        'Matched calibration is deferred. Existing files, if any, are not readiness evidence; '
        'Notebook 07 requires the authenticated fold-excluded post-hoc monotone-Platt v5 protocol.'
    )
if RUN_MATCHED_CALIBRATION_AUDIT and not all(path.is_file() and path.stat().st_size > 0 for path in matched_outputs):
    raise RuntimeError('Matched calibration audit did not produce every required artifact.')


## Summarize Metric Files

In [ ]:
CANONICAL_METRIC_JSON = Path('reports/revision/metrics/calibration_ci_oof_final_ema_predictions.json')
ignored_metric_jsons = sorted(
    path for path in Path('reports/revision/metrics').glob('calibration_ci_*.json')
    if path != CANONICAL_METRIC_JSON
)
if ignored_metric_jsons:
    print('Ignoring non-canonical calibration files:')
    for path in ignored_metric_jsons:
        print(' -', path)
metric_jsons = [CANONICAL_METRIC_JSON] if CANONICAL_METRIC_JSON.exists() else []
if not metric_jsons:
    raise FileNotFoundError(f'Canonical calibration JSON not found: {CANONICAL_METRIC_JSON}')

for path in metric_jsons:
    payload = json.loads(path.read_text(encoding='utf-8'))
    print('\n' + str(path))
    print('dataset:', payload.get('dataset'))
    print('shape:', payload.get('shape'))
    print('metrics:', payload.get('metrics'))
    print('calibration:', payload.get('calibration'))
    print('calibration_micro:', payload.get('calibration_micro'))
    print('reliability:', payload.get('reliability'))
    print('bootstrap_ci:', payload.get('bootstrap_ci'))
    print('artifacts:', payload.get('artifacts'))


## Build Reviewer Tables

In [ ]:
import pandas as pd
from IPython.display import display

table_dir = Path('reports/revision/tables')
table_dir.mkdir(parents=True, exist_ok=True)

calibration_rows = []
bootstrap_rows = []

canonical_metric_json = Path('reports/revision/metrics/calibration_ci_oof_final_ema_predictions.json')
if not canonical_metric_json.exists():
    raise FileNotFoundError(f'Canonical calibration JSON not found: {canonical_metric_json}')
metric_jsons = [canonical_metric_json]
ignored_metric_jsons = sorted(
    path for path in Path('reports/revision/metrics').glob('calibration_ci_*.json')
    if path != canonical_metric_json
)
if ignored_metric_jsons:
    print('Diagnostic/non-canonical calibration files excluded from reviewer tables:')
    for path in ignored_metric_jsons:
        print(' -', path)

for path in metric_jsons:
    payload = json.loads(path.read_text(encoding='utf-8'))
    metrics = payload.get('metrics', {})
    calibration = payload.get('calibration', {})
    calibration_micro = payload.get('calibration_micro', {})
    reliability = payload.get('reliability', {})
    shape = payload.get('shape', {})
    row = {
        'dataset': payload.get('dataset') or Path(payload.get('predictions', path.stem)).stem,
        'predictions': payload.get('predictions'),
        'protocol': payload.get('protocol'),
        'git_commit': payload.get('git_commit'),
        'seed': payload.get('seed'),
        'n_records': shape.get('y_true', [None, None])[0],
        'n_classes': shape.get('y_true', [None, None])[1],
        'threshold': payload.get('threshold'),
        'n_bins': payload.get('n_bins'),
        'n_boot': payload.get('n_boot'),
        'reliability_scope': reliability.get('scope'),
    }
    row.update(metrics)
    row.update(calibration)
    row.update(calibration_micro)
    calibration_rows.append(row)

    for metric_name, ci in payload.get('bootstrap_ci', {}).items():
        bootstrap_rows.append({
            'dataset': row['dataset'],
            'metric': metric_name,
            'mean': ci.get('mean'),
            'ci_low': ci.get('lo'),
            'ci_high': ci.get('hi'),
            'n_boot_valid': ci.get('n_boot_valid'),
            'predictions': row['predictions'],
            'git_commit': row['git_commit'],
        })

table_calibration = table_dir / 'table_calibration.csv'
table_bootstrap = table_dir / 'table_bootstrap_ci.csv'
pd.DataFrame(calibration_rows).to_csv(table_calibration, index=False)
pd.DataFrame(bootstrap_rows).to_csv(table_bootstrap, index=False)

print('Wrote:', table_calibration)
print('Wrote:', table_bootstrap)
print('\nCalibration table preview:')
display(pd.DataFrame(calibration_rows))
print('\nBootstrap CI table preview:')
display(pd.DataFrame(bootstrap_rows))


## Build Reviewer Presentation Assets


In [ ]:
# CPU-only. Builds reliability figure, bootstrap/paired CI compact tables, Q=3 sensitivity table,
# fold-level PCA variance provenance, and explicit normalization/regularization configuration.
RUN_REVIEWER_PRESENTATION_ASSETS = 'auto'
REVIEWER_PRESENTATION_STRICT = True
reviewer_presentation_required = [
    Path('reports/revision/figures/figure_calibration_audit.png'),
    Path('reports/revision/tables/table_calibration_ci_compact.csv'),
    Path('reports/revision/tables/table_paired_baseline_ci_compact.csv'),
    Path('reports/revision/tables/table_pooling_sensitivity_compact.csv'),
    Path('reports/revision/tables/table_fold_pca_provenance.csv'),
    Path('reports/revision/tables/table_training_configuration.csv'),
    Path('reports/revision/tables/table_morphology_transform_contract.csv'),
    Path('reports/revision/manifests/reviewer_completion_input_contract.json'),
]
presentation_oof = Path('reports/revision/predictions/oof_final_ema_predictions.npz')
presentation_freeze = Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json')
presentation_pairs = [
    Path('reports/revision/metrics/paired_full_vs_minirocket_comparison.json'),
    Path('reports/revision/metrics/paired_full_vs_resnet_comparison.json'),
    Path('reports/revision/metrics/paired_full_vs_raw_mamba_comparison.json'),
]
presentation_prerequisites = [
    presentation_oof,
    presentation_freeze,
    Path('reports/revision/metrics/calibration_ci_oof_final_ema_predictions.json'),
    Path('reports/revision/metrics/pooling_sensitivity.csv'),
    Path('reports/revision/manifests/fold_pca_manifest.json'),
    *presentation_pairs,
]
missing_presentation_prerequisites = [str(path) for path in presentation_prerequisites if not path.exists() or path.stat().st_size == 0]
reviewer_assets_ready = all(path.exists() and path.stat().st_size > 0 for path in reviewer_presentation_required)
if reviewer_assets_ready:
    try:
        from scripts.revision.common import sha256_file
        presentation_manifest = json.loads(Path('reports/revision/manifests/reviewer_completion_input_contract.json').read_text(encoding='utf-8'))
        expected_contract = {'oof_sha256': sha256_file(presentation_oof), 'freeze_sha256': sha256_file(presentation_freeze)}
        presentation_runner = Path('scripts/revision/29_reviewer_presentation_assets.py')
        reviewer_assets_ready = (
            presentation_manifest.get('canonical_contract') == expected_contract
            and presentation_manifest.get('runner_sha256') == sha256_file(presentation_runner)
        )
        if not reviewer_assets_ready:
            print('Reviewer presentation assets are stale for the active OOF/freeze contract.')
    except Exception as exc:
        reviewer_assets_ready = False
        print('Reviewer presentation manifest cannot be validated:', repr(exc))
reviewer_assets_should_run = RUN_REVIEWER_PRESENTATION_ASSETS is True or (
    str(RUN_REVIEWER_PRESENTATION_ASSETS).lower() == 'auto' and not reviewer_assets_ready
)
if missing_presentation_prerequisites:
    if RUN_REVIEWER_PRESENTATION_ASSETS is True:
        raise FileNotFoundError('Reviewer presentation assets were forced, but prerequisite artifacts are missing: ' + '; '.join(missing_presentation_prerequisites))
    reviewer_assets_should_run = False
    print('Deferred reviewer presentation assets until Notebook 04 paired comparisons and prerequisite artifacts are current.')
if reviewer_assets_should_run:
    command = 'python -u scripts/revision/29_reviewer_presentation_assets.py --expected-checkpoint-kind final_ema'
    if REVIEWER_PRESENTATION_STRICT:
        command += ' --strict'
    run(command, log_path='reports/revision/logs/reviewer_presentation_assets.log')
else:
    print('Reusing verified reviewer presentation assets.')
for path in reviewer_presentation_required:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/calibration_artifact_inventory.log',
)


## Mirror Metrics To Stable Drive Cache

In [ ]:
source_root = Path('reports/revision')
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/calibration_mirror_publish.log',
)
